In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [3]:
df = pd.read_csv('../data/processed/dataset_long.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
df.head()

,datetime,underlying_price,contract,iv,option_type,strike,hour,minute,day_of_week,date,session_progress,days_to_expiry,moneyness,log_moneyness,dist_from_atm,dist_from_atm_pct,is_ce,strike_rank,iv_neighbor_-2,iv_neighbor_-1,iv_neighbor_+1,iv_neighbor_+2,iv_neighbor_mean,wide_iv_neighbor_mean,iv_lag_1,iv_roll_mean_5,iv_roll_std_5,iv_roll_mean_10,mean_ce_iv,mean_pe_iv
0,2026-01-07 09:15:00,26111.65,NIFTY27JAN2625200CE,0.12662,CE,25200,9,15,2,2026-01-07,4860.0,20.2569,0.965086,-0.035538,911.65,0.034914,1,0.0,NaN,NaN,0.12330,0.11741,0.12330,0.11741,NaN,NaN,NaN,NaN,0.102447,0.102447
1,2026-01-07 09:20:00,26141.40,NIFTY27JAN2625200CE,0.08632,CE,25200,9,20,2,2026-01-07,4865.0,20.2535,0.963988,-0.036676,941.40,0.036012,1,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.12662,NaN,NaN,NaN,0.098723,0.098723
2,2026-01-07 09:25:00,26139.35,NIFTY27JAN2625200CE,0.09147,CE,25200,9,25,2,2026-01-07,4870.0,20.2500,0.964064,-0.036598,939.35,0.035936,1,0.0,NaN,NaN,NaN,0.09514,NaN,0.09514,0.08632,0.106470,0.028496,NaN,0.091074,0.091074
3,2026-01-07 09:30:00,26128.95,NIFTY27JAN2625200CE,0.10860,CE,25200,9,30,2,2026-01-07,4875.0,20.2465,0.964447,-0.036200,928.95,0.035553,1,0.0,NaN,NaN,0.10842,0.11150,0.10842,0.11150,0.09147,0.101470,0.021932,0.101470,0.103252,0.103252
4,2026-01-07 09:35:00,26131.90,NIFTY27JAN2625200CE,0.10462,CE,25200,9,35,2,2026-01-07,4880.0,20.2431,0.964339,-0.036313,931.90,0.035661,1,0.0,NaN,NaN,0.10538,0.12459,0.10538,0.12459,0.10860,0.103252,0.018259,0.103252,0.104097,0.104097


In [4]:
df.columns

Index(['datetime', 'underlying_price', 'contract', 'iv', 'option_type',
       'strike', 'hour', 'minute', 'day_of_week', 'date', 'session_progress',
       'days_to_expiry', 'moneyness', 'log_moneyness', 'dist_from_atm',
       'dist_from_atm_pct', 'is_ce', 'strike_rank', 'iv_neighbor_-2',
       'iv_neighbor_-1', 'iv_neighbor_+1', 'iv_neighbor_+2',
       'iv_neighbor_mean', 'wide_iv_neighbor_mean', 'iv_lag_1',
       'iv_roll_mean_5', 'iv_roll_std_5', 'iv_roll_mean_10', 'mean_ce_iv',
       'mean_pe_iv'],
      dtype='str')

In [20]:
dates = sorted(df['datetime'].dt.date.unique())

folds = [
    {'name': 'Fold 1', 'train_dates': dates[:9],  'val_dates': dates[9:11]},
    {'name': 'Fold 2', 'train_dates': dates[:10], 'val_dates': dates[10:12]},
]

In [7]:
def make_holdout(val_df,random_state=2):
    np.random.seed(42)

    available_rows = val_df[val_df['iv'].notna()].index

    hidden_rows = np.random.choice(available_rows,size=int(0.2*len(available_rows)),replace=False)

    ground_truth = val_df.loc[hidden_rows,'iv'].values

    val_masked = val_df.copy()

    val_masked.loc[hidden_rows,'iv'] = np.nan

    return val_masked,hidden_rows,ground_truth

In [8]:
# evaluation functions for ml models
def eval_pred(actual,predicted):
    valid = ~np.isnan(predicted)

    mse = mean_squared_error(actual[valid],predicted[valid])
    coverage = valid.mean()

    print(f"MSE: {mse:6f}")
    print(f"Coverage: {coverage:6f}")

    return mse,coverage

In [9]:
import xgboost
import lightgbm
import catboost

### Model 1: XGBoost will all features

In [21]:
from xgboost import XGBRegressor

fold_mses, fold_coverages, feature_importances = [],[],[]

drop_cols = ['iv','datetime','contract','option_type','date']

for fold in folds:
    train_df = df[df['datetime'].dt.date.isin(fold['train_dates'])].copy()

    val_df = df[df['datetime'].dt.date.isin(fold['val_dates'])].copy()

    val_masked, hidden_rows, ground_truth = make_holdout(val_df)

    # train data
    train_obs = train_df[train_df['iv'].notna()].copy()

    X_train = train_obs.drop(columns=drop_cols)
    y_train = train_obs['iv']

    # validation data
    X_val = val_masked.loc[hidden_rows].drop(columns=drop_cols)

    xgb_model = XGBRegressor(
        n_estimators= 500,
        learning_rate= 0.05,
        max_depth= 6,
        subsample= 0.8,
        colsample_bytree= 0.8,
        random_state= 2,
    )

    xgb_model.fit(X_train,y_train)
    xgb_pred = xgb_model.predict(X_val)

    xgb_mse, xgb_coverage = eval_pred(ground_truth,xgb_pred)

    fold_mses.append(xgb_mse)
    fold_coverages.append(xgb_coverage)
    feature_importances.append(xgb_model.feature_importances_)

print(f"xgb_mse: {np.mean(fold_mses):6f}")
print(f"xgb_coverage: {np.mean(fold_coverages):6f}")
print(f"feature importances: {feature_importances}")


MSE: 0.000060
Coverage: 1.000000
MSE: 0.000047
Coverage: 1.000000
xgb_mse: 0.000054
xgb_coverage: 1.000000
feature importances: [array([0.00363282, 0.00907561, 0.00155192, 0.00223305, 0.00444412,
       0.00367086, 0.00336628, 0.00785213, 0.00631918, 0.00849222,
       0.00983764, 0.        , 0.00690801, 0.0082335 , 0.00724757,
       0.01690204, 0.01307639, 0.5044574 , 0.05791805, 0.00531302,
       0.2592039 , 0.00362226, 0.0293275 , 0.01106107, 0.01625342],
      dtype=float32), array([0.00247129, 0.00621403, 0.00108871, 0.00110867, 0.00310131,
       0.00215398, 0.00219415, 0.02390117, 0.00368832, 0.0064261 ,
       0.00643909, 0.0005458 , 0.00393209, 0.00472965, 0.00666619,
       0.0132345 , 0.00542649, 0.26722205, 0.03958256, 0.00331858,
       0.49992165, 0.00180436, 0.07908553, 0.00519456, 0.01054917],
      dtype=float32)]


In [22]:
avg_importance = np.mean(feature_importances,axis=0)

importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': avg_importance
})

importance_df = importance_df.sort_values(by='importance',ascending=False)

importance_df

,feature,importance
17,iv_neighbor_mean,0.385840
20,iv_roll_mean_5,0.379563
22,iv_roll_mean_10,0.054207
18,wide_iv_neighbor_mean,0.048750
7,moneyness,0.015877
15,iv_neighbor_+1,0.015068
24,mean_pe_iv,0.013401
16,iv_neighbor_+2,0.009251
10,dist_from_atm_pct,0.008138
23,mean_ce_iv,0.008128


In [24]:
selected_features = importance_df[importance_df['importance']>=0.005]['feature'].tolist()

selected_features

['iv_neighbor_mean',
 'iv_roll_mean_5',
 'iv_roll_mean_10',
 'wide_iv_neighbor_mean',
 'moneyness',
 'iv_neighbor_+1',
 'mean_pe_iv',
 'iv_neighbor_+2',
 'dist_from_atm_pct',
 'mean_ce_iv',
 'strike',
 'dist_from_atm',
 'iv_neighbor_-1',
 'iv_neighbor_-2',
 'strike_rank',
 'log_moneyness']

In [25]:
df_pruned = df[selected_features+['iv','contract','option_type','date','datetime']].copy()
print(df_pruned.shape)
df_pruned.head()

(27300, 21)


,iv_neighbor_mean,iv_roll_mean_5,iv_roll_mean_10,wide_iv_neighbor_mean,moneyness,iv_neighbor_+1,mean_pe_iv,iv_neighbor_+2,dist_from_atm_pct,mean_ce_iv,strike,dist_from_atm,iv_neighbor_-1,iv_neighbor_-2,strike_rank,log_moneyness,iv,contract,option_type,date,datetime
0,0.12330,NaN,NaN,0.11741,0.965086,0.12330,0.102447,0.11741,0.034914,0.102447,25200,911.65,NaN,NaN,0.0,-0.035538,0.12662,NIFTY27JAN2625200CE,CE,2026-01-07,2026-01-07 09:15:00
1,NaN,NaN,NaN,NaN,0.963988,NaN,0.098723,NaN,0.036012,0.098723,25200,941.40,NaN,NaN,0.0,-0.036676,0.08632,NIFTY27JAN2625200CE,CE,2026-01-07,2026-01-07 09:20:00
2,NaN,0.106470,NaN,0.09514,0.964064,NaN,0.091074,0.09514,0.035936,0.091074,25200,939.35,NaN,NaN,0.0,-0.036598,0.09147,NIFTY27JAN2625200CE,CE,2026-01-07,2026-01-07 09:25:00
3,0.10842,0.101470,0.101470,0.11150,0.964447,0.10842,0.103252,0.11150,0.035553,0.103252,25200,928.95,NaN,NaN,0.0,-0.036200,0.10860,NIFTY27JAN2625200CE,CE,2026-01-07,2026-01-07 09:30:00
4,0.10538,0.103252,0.103252,0.12459,0.964339,0.10538,0.104097,0.12459,0.035661,0.104097,25200,931.90,NaN,NaN,0.0,-0.036313,0.10462,NIFTY27JAN2625200CE,CE,2026-01-07,2026-01-07 09:35:00


In [26]:
df_pruned.to_csv('../data/processed/dataset_long_pruned.csv',index=False)

### Model 2: XGBoost with pruned features

In [27]:
fold_mses, fold_coverages = [],[]

drop_cols = ['iv','datetime','contract','option_type','date']

for fold in folds:
    train_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['train_dates'])].copy()

    val_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['val_dates'])].copy()

    val_masked, hidden_rows, ground_truth = make_holdout(val_df)

    # train data
    train_obs = train_df[train_df['iv'].notna()].copy()

    X_train = train_obs.drop(columns=drop_cols)
    y_train = train_obs['iv']

    # validation data
    X_val = val_masked.loc[hidden_rows].drop(columns=drop_cols)

    xgb_model = XGBRegressor(
        n_estimators= 500,
        learning_rate= 0.05,
        max_depth= 6,
        subsample= 0.8,
        colsample_bytree= 0.8,
        random_state= 2,
    )

    xgb_model.fit(X_train,y_train)
    xgb_pred = xgb_model.predict(X_val)

    xgb_mse, xgb_coverage = eval_pred(ground_truth,xgb_pred)

    fold_mses.append(xgb_mse)
    fold_coverages.append(xgb_coverage)

print(f"xgbp_mse: {np.mean(fold_mses):6f}")
print(f"xgbp_coverage: {np.mean(fold_coverages):6f}")

MSE: 0.000044
Coverage: 1.000000
MSE: 0.000035
Coverage: 1.000000
xgbp_mse: 0.000040
xgbp_coverage: 1.000000


### Model 3: LightGBM

In [28]:
from lightgbm import LGBMRegressor

fold_mses, fold_coverages = [],[]

drop_cols = ['iv','datetime','contract','option_type','date']

for fold in folds:
    train_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['train_dates'])].copy()

    val_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['val_dates'])].copy()

    val_masked, hidden_rows, ground_truth = make_holdout(val_df)

    # train data
    train_obs = train_df[train_df['iv'].notna()].copy()

    X_train = train_obs.drop(columns=drop_cols)
    y_train = train_obs['iv']

    # validation data
    X_val = val_masked.loc[hidden_rows].drop(columns=drop_cols)

    lgbm_model = LGBMRegressor(
        n_estimators= 500,
        learning_rate= 0.05,
        max_depth= 6,
        subsample= 0.8,
        colsample_bytree= 0.8,
        random_state= 2,
    )

    lgbm_model.fit(X_train,y_train)
    lgbm_pred = lgbm_model.predict(X_val)

    lgbm_mse, lgbm_coverage = eval_pred(ground_truth,lgbm_pred)

    fold_mses.append(lgbm_mse)
    fold_coverages.append(lgbm_coverage)

print(f"MSE from both folds: {fold_mses}")
print(f"lgbm_mse: {np.mean(fold_mses):6f}")
print(f"lgbm_coverage: {np.mean(fold_coverages):6f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008621 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3597
[LightGBM] [Info] Number of data points in the train set: 15192, number of used features: 16
[LightGBM] [Info] Start training from score 0.108086
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

### Model 4: CatBoost

In [29]:
from catboost import CatBoostRegressor

fold_mses, fold_coverages = [],[]

drop_cols = ['iv','datetime','contract','option_type','date']

for fold in folds:
    train_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['train_dates'])].copy()

    val_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['val_dates'])].copy()

    val_masked, hidden_rows, ground_truth = make_holdout(val_df)

    # train data
    train_obs = train_df[train_df['iv'].notna()].copy()

    X_train = train_obs.drop(columns=drop_cols)
    y_train = train_obs['iv']

    # validation data
    X_val = val_masked.loc[hidden_rows].drop(columns=drop_cols)

    cb_model = CatBoostRegressor(
        n_estimators= 500,
        learning_rate= 0.05,
        max_depth= 6,
        subsample= 0.8,
        random_state= 2,
    )

    cb_model.fit(X_train,y_train)
    cb_pred = cb_model.predict(X_val)

    cb_mse, cb_coverage = eval_pred(ground_truth,cb_pred)

    fold_mses.append(cb_mse)
    fold_coverages.append(cb_coverage)

print(f"MSE from both folds: {fold_mses}")
print(f"cb_mse: {np.mean(fold_mses):6f}")
print(f"cb_coverage: {np.mean(fold_coverages):6f}")

0:	learn: 0.0129226	total: 10ms	remaining: 4.99s
1:	learn: 0.0124073	total: 15.9ms	remaining: 3.96s
2:	learn: 0.0119148	total: 23ms	remaining: 3.81s
3:	learn: 0.0114542	total: 27.1ms	remaining: 3.36s
4:	learn: 0.0110246	total: 33.3ms	remaining: 3.29s
5:	learn: 0.0106173	total: 37.6ms	remaining: 3.1s
6:	learn: 0.0102425	total: 42.4ms	remaining: 2.98s
7:	learn: 0.0098614	total: 46.2ms	remaining: 2.84s
8:	learn: 0.0095130	total: 50.2ms	remaining: 2.74s
9:	learn: 0.0091722	total: 54.3ms	remaining: 2.66s
10:	learn: 0.0088422	total: 58.6ms	remaining: 2.61s
11:	learn: 0.0085460	total: 62.8ms	remaining: 2.56s
12:	learn: 0.0082627	total: 66.5ms	remaining: 2.49s
13:	learn: 0.0079828	total: 70ms	remaining: 2.43s
14:	learn: 0.0077328	total: 73.7ms	remaining: 2.38s
15:	learn: 0.0074874	total: 76.6ms	remaining: 2.32s
16:	learn: 0.0072592	total: 79.3ms	remaining: 2.25s
17:	learn: 0.0070322	total: 83.6ms	remaining: 2.24s
18:	learn: 0.0068237	total: 86.7ms	remaining: 2.19s
19:	learn: 0.0066303	total: 8